# 08. DeepSeek LangChain MCP Client

## 01. 安装依赖

In [4]:

%%capture #避免显示pip安装信息
%pip install langchain-mcp-adapters langchain-deepseek python-dotenv langgraph

## 2. 自定义MCP Server

> 参考：[langchain-mcp-adapters](https://github.com/langchain-ai/langchain-mcp-adapters)

### 2.1 安装依赖

In [7]:
%%capture #避免显示pip安装信息
%pip install fastmcp

### 2.2 核心代码

In [ ]:
# math_server.py 见:./math_server.py，这里暂不支持执行
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")

## Client 客户端

In [2]:
# Create server parameters for stdio connection
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent

from langchain_deepseek import ChatDeepSeek

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

model = ChatDeepSeek(model="deepseek-chat")

server_params = StdioServerParameters(
    command="python",
    # Make sure to update to the full absolute path to your math_server.py file
    args=["./math_server.py"],
)

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        # Initialize the connection
        await session.initialize()

        # Get tools
        tools = await load_mcp_tools(session)

        # Create and run the agent
        agent = create_react_agent(model, tools)
        agent_response = await agent.ainvoke({"messages": "what's (3 + 5) x 12?"})
        print("Answer:", agent_response)

Answer: {'messages': [HumanMessage(content="what's (3 + 5) x 12?", additional_kwargs={}, response_metadata={}, id='c2533d58-dc51-43c1-9bfa-515ef22217c1'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_0587925b-8405-455b-92ab-058b79be58fc', 'function': {'arguments': '{"a": 3, "b": 5}', 'name': 'add'}, 'type': 'function', 'index': 0}, {'id': 'call_1_ed14c7eb-fc77-4fc2-b1ad-f838f9b6caec', 'function': {'arguments': '{"a": 8, "b": 12}', 'name': 'multiply'}, 'type': 'function', 'index': 1}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 218, 'total_tokens': 266, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 26}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3d5141a69a_prod0225', 'id': 'df64c236-87df-4848-97a5-d6bb7d5476c9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-73592cd2-

## 3. 多个 MCP 服务器

In [ ]:
from typing import List
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Weather")

@mcp.tool()
async def get_weather(location: str) -> str:
    """Get weather for location."""
    return "It's always sunny in New York"

if __name__ == "__main__":
    mcp.run(transport="sse")

## 4. 多个MCP 集成Client

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat")

async with MultiServerMCPClient(
    {
        "math": {
            "command": "python",
            # Make sure to update to the full absolute path to your math_server.py file
            "args": ["./math_server.py"],
            "transport": "stdio",
        },
        "weather": {
            # make sure you start your weather server on port 8000
            "url": "http://localhost:8000/sse",
            "transport": "sse",
        }
    }
) as client:
    agent = create_react_agent(model, client.get_tools())
    math_response = await agent.ainvoke({"messages": "what's (3 + 5) x 12?"})
    print(math_response)
    weather_response = await agent.ainvoke({"messages": "what is the weather in nyc?"})
    print(weather_response)

{'messages': [HumanMessage(content="what's (3 + 5) x 12?", additional_kwargs={}, response_metadata={}, id='ba98248d-aa79-45ac-a90b-fee5395c4981'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_7bf5d54e-9c59-44bd-9136-da7c49717200', 'function': {'arguments': '{"a": 3, "b": 5}', 'name': 'add'}, 'type': 'function', 'index': 0}, {'id': 'call_1_5c3c8c75-3981-4c9f-9f58-a816b47d7de6', 'function': {'arguments': '{"a": 8, "b": 12}', 'name': 'multiply'}, 'type': 'function', 'index': 1}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 296, 'total_tokens': 344, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 104}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3d5141a69a_prod0225', 'id': '6c0e51f1-cc9f-4282-8c26-2a69d1e9fb83', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-f6eaf98d-e8f5-40

## 04. 对接 mcp.so

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

from langchain_deepseek import ChatDeepSeek
import os
from dotenv import load_dotenv

load_dotenv()

model = ChatDeepSeek(model="deepseek-chat")

async with MultiServerMCPClient(
    {
        "tavily-mcp": {
            "command": "npx",
            "args": [
                "-y",
                "tavily-mcp@0.1.4"
            ],
            "env": {
                "TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")
            },
            "autoApprove": [],
            "name": "tavily-mcp"
        }
    }
) as client:
    agent = create_react_agent(model, client.get_tools())
    weather_response = await agent.ainvoke({"messages": "what is the weather in nyc?"})
    print(weather_response)

{'messages': [HumanMessage(content='what is the weather in nyc?', additional_kwargs={}, response_metadata={}, id='bcd68f6d-5aca-4375-8230-876a56d3e964'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_69cb055e-39bf-4801-9a81-832529e8d2e4', 'function': {'arguments': '{"query":"current weather in New York City","search_depth":"basic","topic":"general","max_results":1}', 'name': 'tavily-search'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 999, 'total_tokens': 1038, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 960}, 'prompt_cache_hit_tokens': 960, 'prompt_cache_miss_tokens': 39}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3d5141a69a_prod0225', 'id': 'f9c12cb5-c2e5-4c90-a50d-6a4a179e2fb3', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-176b6b62-d8bc-4fdb-acef-762a17606dc2-0', tool_calls=[{'name': 'tavily